In [ ]:
import pandas as pd
import csv
import os

In [ ]:
edu_fod_delete_first_row = {
    "Tab1_FOD.xlsx": True,
    "Tab2_FOD.xlsx": False,
    "Tab3_FOD.xlsx": False,
    "Tab4_FOD.xlsx": False,
    "Tab5_FOD.xlsx": False,
    "Tab6_FOD.xlsx": True,
    "Tab7_FOD.xlsx": True
}

# This dictionary specifies which tables need the first row dropped.

In [ ]:
edu_fod_file_pairs = [
    ("Tab1_FOD.xlsx", "fod_demo_soc.xlsx"),
    ("Tab2_FOD.xlsx", "fod_earn_sex.xlsx"),
    ("Tab3_FOD.xlsx", "fod_earn_age.xlsx"),
    ("Tab4_FOD.xlsx", "fod_earn_race.xlsx"),
    ("Tab5_FOD.xlsx", "fod_earn_edu_att.xlsx"),
    ("Tab6_FOD.xlsx", "fod_metro.xlsx"),
    ("Tab7_FOD.xlsx", "fod_earn_metro.xlsx")
]

# This list of tuples was created to show the input (original) and output (new file) of the data files being renamed. 

In [ ]:
def get_column_names(df_header: pd.DataFrame, drop_first_row: bool = False) -> dict[str, str]:
    headers = df_header.head(3)
    
    if drop_first_row:
        headers = headers.drop(0)

    column_names = {}
    last_value = ''
    
    for column in headers:
        first_row_value = headers[column].iloc[0]
        
        if not pd.isna(first_row_value):
            last_value = str(first_row_value).strip()

        second_row_value = headers[column].iloc[1]
        
        if not pd.isna(second_row_value) and second_row_value != "":
            second_row_value = str(second_row_value).strip()
            result = f"{last_value}_{second_row_value}"
            # result = f"{second_row_value}_{last_value}"
        else:
            result = last_value
        
        column_names[column] = result
    
    return column_names

In [ ]:
def rename_all_sheets(all_sheets: dict, drop_first_row: bool = False) -> dict[str, pd.DataFrame]:
    
    renamed_sheets = {}
    
    for sheet_name, df in all_sheets.items():
        names = get_column_names(df, drop_first_row=drop_first_row)

        # getting rid of the old file names
        if drop_first_row:
            df = df.drop([0, 1, 2])
        else:
            df = df.drop([0, 1])
        
        renamed_sheets[sheet_name] = df.rename(columns=names)
        
    return renamed_sheets

In [ ]:
def write_excel_file(all_sheets: dict[str, pd.DataFrame], output_file: str):
    
    with pd.ExcelWriter(output_file) as writer:
        for sheet_name, df in all_sheets.items():
            df.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
def process_files(file_pairs: list[tuple[str, str]], delete_first_row: dict[str, bool], input_dir: str, output_dir: str):
    
    for input_name, output_name in file_pairs:
        input_file = f"{input_dir}/{input_name}"
        output_file = f"{output_dir}/{output_name}"

        all_sheets = pd.read_excel(input_file, sheet_name=None) # reads all sheets

        renamed_sheets = rename_all_sheets(all_sheets, drop_first_row=[input_name]) # renames all sheets

        return renamed_sheets

        write_excel_file(renamed_sheets, output_file) # writes new file

        print(f"Copied {input_name} / {output_file}")

In [ ]:
def strip_object_columns(df: pd.DataFrame) -> pd.DataFrame:

    for column in df.select_dtypes(include=['object']).columns:
        df[column] = df[column].astype(str).str.strip()
    
    return df

In [ ]:
process_files(
    edu_fod_file_pairs,
    edu_fod_delete_first_row,
    input_dir="data/education/original",
    output_dir="data/education/working"
)

In [ ]:
def clean_numeric_columns(df: pd.DataFrame, threshold: float = 0.7) -> pd.DataFrame: # adds a threshold of 0.7 (70%); so unless the row has 70% >= non-null values that are numeric, it will execute, if not, no conversion will be done on that row 
    df = df.copy()
    
    for col in df.columns:
        if col.lower() in {"field_of_degree"}:
            continue
        
        cleaned = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
        )

        numeric = pd.to_numeric(cleaned, errors="coerce") # updated "ignore" to "coerce" due to FutureWarning

        non_null = cleaned.notna().sum()
        numeric_count = numeric.notna().sum()

        if non_null > 0 and numeric_count / non_null >= threshold: 
            df[col] = numeric
    
    return df

In [ ]:
def remove_empty_rows(df: pd.DataFrame) -> pd.DataFrame: # this will remove the rows where the numeric columns are NaN
    numeric_cols = df.select_dtypes(include=['number']).columns
    
    if len(numeric_cols) == 0:
        return df # this checks the number of numeric rows; if 0, then it will end
    
    return df.dropna(subset=numeric_cols, how='all')

In [ ]:
def format_numbers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    def format_whole_numbers(x):
        if pd.isna(x):
            return x
        if isinstance(x, float) and x.is_integer():
            return int(x)
        return x

    numeric_cols = df.select_dtypes(include=['number']).columns
    df[numeric_cols] = df[numeric_cols].applymap(format_whole_numbers)

    return df

# function that formats numbers with no decimal value to whole integers and leaves float values untouched

In [ ]:
def batch_clean_csv(
    input_dir: str,
    output_dir: str,
    filenames: list[str]
    ):
    
    for filename in filenames:
        input_file = f"{input_dir}/{filename}"
        output_file = f"{output_dir}/{filename}"

        df = pd.read_csv(input_file)    

    df = strip_object_columns(df)
    df = clean_numeric_columns(df)
    df = remove_empty_rows(df)
    df = format_numbers(df)

    base, ext = filename.rsplit(".", 1) # # splits above and adds 'base' (everything before the period) and 'ext' (everything after the period)
    output_file = f"{output_dir}/{base}_cleaned.{ext}" # splits above and adds 'base' (everything before the period) and 'ext' (everything after the period); adds '_cleaned' to the new file name

    df.to_csv(output_file, index=False)
    print(f"Saved cleaned CSV: {output_file}")

In [ ]:
csv_files = [
    "fobd_by_state.csv"
    ]

batch_clean_csv(
    input_dir="data/education/working",
    output_dir="data/education/cleaned",
    filenames=csv_files
    )

In [ ]:
def rename_csv_files_cleaned(filenames: list[str], output_dir: str, suffix: str = "_cleaned"):

    for file_path in filenames:
        df = pd.read_csv(file_path)

        base_name = file_path.split("/")[-1] # get file name
        name, ext = base_name.rsplit(".", 1) # "file.csv" -> "file", "csv"

        new_filename = f"{name}{suffix}.{ext}"
        output_path = f"{output_dir}/{new_filename}"

        # df.to_csv(output_path, index=False)
        # print(f"Renamed: {base_name} to {new_filename}")

In [ ]:
csv_rename_files = [
    "fod_demo_soc_table"
]

In [ ]:
def clean_test(df: pd.DataFrame, use_dot_hierarchy: bool = True) -> pd.DataFrame:
    df = df.copy()

    if use_dot_hierarchy:
        df.columns =df.columns.str.replace("_", ".", regex=False)

    df = strip_object_columns(df)
    df = clean_numeric_columns(df)
    df = df.dropna(how='all')

    return df

In [ ]:
def excel_to_csv_batch(
        input_dir: str,
        output_dir: str,
        filenames: list[str],
        use_dot_hierarchy: bool = True # preferred method of header hierarchy # can toggle True/False
):

    for filename in filenames:
        input_file = f"{input_dir}/{filename}"
        orig_name = filename.replace(".xlsx", "")

        all_sheets = pd.read_excel(input_file, sheet_name=None)

        for sheet_name, df in all_sheets.items():
            safe_sheet_name = sheet_name.replace(" ", "_").lower()
            
            if use_dot_hierarchy:
                df.columns = df.columns.str.replace("_", ".", regex=False) # replacing "_" with "." for header hierarchy
            
            df = strip_object_columns(df)

            df = clean_numeric_columns(df)

            df = remove_empty_rows(df)

            df = format_numbers(df)
            
            output_file = f"{output_dir}/{orig_name}_{safe_sheet_name}.csv"

            df.to_csv(output_file, index=False)
            print(f"Saved: {output_file}")

In [ ]:
edu_files = [
    "fod_demo_soc.xlsx",
    "fod_earn_age.xlsx",
    "fod_earn_edu_att.xlsx",
    "fod_earn_metro.xlsx",
    "fod_earn_race.xlsx",
    "fod_earn_sex.xlsx",
    "fod_metro.xlsx"
]

excel_to_csv_batch(
    input_dir="data/education/working",
    output_dir="data/education/cleaned",
    filenames=edu_files
)

In [ ]:
FORMAT_MAP = {}

In [ ]:
copy_csv = False # flag variable used to keep from constant overwriting when using "Run All" # can be changed to 'True' for execution.

input_file = "data/child_care/original/Map by price and county_Full Data_data.csv"
output_file = "data/child_care/cleaned/child_care_full_map.csv"

if copy_csv:
    df_childcare_map = pd.read_csv(input_file, dtype={"County Fips Code": str}) # have to convert to str for pandas to read FIPS that start with 0 correctly
    
    df_childcare_map = pd.rename(columns={
        "Flfpr 20To64: Women's Labor Force Participation Rate (Percentage)",
        "Fme 2022: Women's Median Earnings",
        "Hispanic: Hispanic (Percentage)",
        


    }, inplace=True)
    
    df_childcare_map["County Fips Code"] = df_childcare_map["County Fips Code"].str.zfill(5)
    
    df_childcare_map.to_csv(output_file, index=False)

# df_childcare_map = pd.read_csv(output_file, dtype={"County Fips Code": str}) # have to specify dtype when reading, or will show incorrect FIPS starting with 0
# print(df_childcare_map.head())
# print(df_childcare_map.columns)
# df_childcare_map.info()